# An upper bound on one labeller's landmark noise (T6)

`HANDOFF.md` §5 item 1, from the P1 corpus with no new labels. Estimator and its
justification are in `fishsense_cscw/landmark.py`'s docstring — in short, the handoff's
"spread of the best-presented frames" is biased *low*; the upper half-width about the
cell median is the version that errs high.

In [1]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.precision", 2)

from fishsense_cscw import landmark as lm
df = lm.load_cohort()
cells = lm.noise_cells(df, bin_m=0.25, min_frames=6)
print(f"{len(df)} frames; {len(cells)} usable cells over {cells.dive_id.nunique()} dives, {int(cells.n.sum())} frames")
cells.groupby("model").agg(cells=("n", "size"), frames=("n", "sum"), range_m=("range_m", "median"),
                           fish_px=("length_px", "median"), bound_pct=("bound_pct", "median"),
                           loose_pct=("loose_pct", "median"), px_per_endpoint=("px_per_endpoint", "median")
                           ).sort_values("frames", ascending=False)

1051 frames; 39 usable cells over 6 dives, 329 frames


,cells,frames,range_m,fish_px,bound_pct,loose_pct,px_per_endpoint
model,,,,,,,
Box,13,101,1.88,223.70,0.55,0.52,0.96
Snook,8,64,1.74,698.52,1.26,2.00,5.91
Weasly Fish,7,60,1.36,613.55,0.45,0.41,1.67
Grouper,5,52,1.44,673.87,0.60,0.92,2.77
Purple Angel,3,31,1.65,330.67,0.51,0.50,1.19
Shark,2,15,1.43,1210.11,0.74,0.76,6.36
Ruler,1,6,1.86,505.16,0.38,0.57,1.36


In [2]:
rows = []
for b in (0.15, 0.25, 0.5):
    for m in (5, 6, 8):
        c = lm.noise_cells(df, b, m)
        rows.append(dict(bin_m=b, min_frames=m, cells=len(c), dives=c.dive_id.nunique(), frames=int(c.n.sum()),
                         bound_pct=c.bound_pct.median(), px_per_endpoint=c.px_per_endpoint.median()))
pd.DataFrame(rows)

,bin_m,min_frames,cells,dives,frames,bound_pct,px_per_endpoint
0,0.15,5,25,5,192,0.67,2.74
1,0.15,6,25,5,192,0.67,2.74
2,0.15,8,11,5,102,0.59,2.00
3,0.25,5,39,6,329,0.56,1.99
4,0.25,6,39,6,329,0.56,1.99
5,0.25,8,23,6,225,0.56,2.03
6,0.50,5,81,19,732,0.75,2.08
7,0.50,6,81,19,732,0.75,2.08
8,0.50,8,43,12,485,0.71,2.40


## Result

**One labeller's combined landmark + ranging noise on rigid targets is at most ~0.6 %
of length — about 2 px per endpoint**, stable at 0.56–0.75 % across every binning
choice. The 0.5 m bins reach all 19 dives and read slightly higher, as they should: wider
bins admit more range drift into a bound that errs high.

What it is and is not:

- It is **intra-labeller consistency on rigid models** with crisp landmarks. It is *not*
  the inter-annotator ceiling (T2, T21) and not live-fish difficulty.
- Against it, the detector's median snout error (1.3–2.7 %, T4) is 2–4× this bound and
  its fork error (4.8–9 %) is roughly 10×. The detector is nowhere near one labeller's
  own consistency — which is the comparison the paper can make today.
- On P1's near-range Weasly Fish bias (−6.8 % below 0.8 m): the one near-range cell has
  a spread of ~0.24 %, so **random** landmark noise cannot produce it. If it is a
  landmark effect it is a systematic placement convention, not noise. One cell — weak.